# Microgrid - In-Class Example 2 (off-grid)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)


In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    Binary, NonNegativeReals, minimize, value
)

# ---- Data: loaded from external file 'MG_IC_e2_data.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('MG_IC_e2_data.txt')

GEN_data     = _d['GEN']
PERIOD_data  = _d['PERIOD']
STO_data     = _d['STORAGE']
gen_min      = _d['gen_min']
gen_max      = _d['gen_max']
gen_RRlimit  = _d['gen_RRlimit']
gen_OpCost   = _d['gen_OpCost']
gen_NlCost   = _d['gen_NlCost']
gen_SuCost   = _d['gen_SuCost']
ES_EInit     = _d['ES_EInit']
ES_Emax      = _d['ES_Emax']
ES_Emin      = _d['ES_Emin']
ES_PmaxC     = _d['ES_PmaxC']
ES_PmaxD     = _d['ES_PmaxD']
ES_EffiC     = _d['ES_EffiC']
ES_EffiD     = _d['ES_EffiD']
Time_TotalPd = _d['Time_TotalPd']
SolarP       = _d['SolarP']
m = ConcreteModel()
m.GEN     = Set(initialize=GEN_data,    ordered=True)
m.PERIOD  = Set(initialize=PERIOD_data, ordered=True)
m.STORAGE = Set(initialize=STO_data,    ordered=True)

m.gen_min     = Param(m.GEN, initialize=gen_min)
m.gen_max     = Param(m.GEN, initialize=gen_max)
m.gen_RRlimit = Param(m.GEN, initialize=gen_RRlimit)
m.gen_OpCost  = Param(m.GEN, initialize=gen_OpCost)
m.gen_NlCost  = Param(m.GEN, initialize=gen_NlCost)
m.gen_SuCost  = Param(m.GEN, initialize=gen_SuCost)

m.ES_EInit = Param(m.STORAGE, initialize=ES_EInit)
m.ES_Emax  = Param(m.STORAGE, initialize=ES_Emax)
m.ES_Emin  = Param(m.STORAGE, initialize=ES_Emin)
m.ES_PmaxC = Param(m.STORAGE, initialize=ES_PmaxC)
m.ES_PmaxD = Param(m.STORAGE, initialize=ES_PmaxD)
m.ES_EffiC = Param(m.STORAGE, initialize=ES_EffiC)
m.ES_EffiD = Param(m.STORAGE, initialize=ES_EffiD)

m.Time_TotalPd = Param(m.PERIOD, initialize=Time_TotalPd)
m.SolarP       = Param(m.PERIOD, initialize=SolarP)

m.u  = Var(m.GEN, m.PERIOD, domain=Binary)
m.v  = Var(m.GEN, m.PERIOD, bounds=(0,1))
m.Pg = Var(m.GEN, m.PERIOD)

m.u_c_ES = Var(m.STORAGE, m.PERIOD, domain=Binary)
m.u_d_ES = Var(m.STORAGE, m.PERIOD, domain=Binary)
m.P_c_ES = Var(m.STORAGE, m.PERIOD, domain=NonNegativeReals)
m.P_d_ES = Var(m.STORAGE, m.PERIOD, domain=NonNegativeReals)
m.E_ES   = Var(m.STORAGE, m.PERIOD, domain=NonNegativeReals)

m.obj = Objective(
    rule=lambda mm: sum(mm.gen_OpCost[g]*mm.Pg[g,t]
                       + mm.gen_NlCost[g]*mm.u[g,t]
                       + mm.gen_SuCost[g]*mm.v[g,t]
                       for g in mm.GEN for t in mm.PERIOD),
    sense=minimize
)

def pb_rule(mm, t):
    return sum(mm.Pg[g,t] for g in mm.GEN) == mm.Time_TotalPd[t] - mm.SolarP[t] \
           + sum(mm.P_c_ES[e,t] - mm.P_d_ES[e,t] for e in mm.STORAGE)
m.PowerBalance = Constraint(m.PERIOD, rule=pb_rule)

m.genLimit_Min = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.gen_min[g]*mm.u[g,t] <= mm.Pg[g,t])
m.genLimit_Max = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.Pg[g,t] <= mm.gen_max[g]*mm.u[g,t])

def rr_up(mm,g,t):
    if t == mm.PERIOD.first(): return Constraint.Skip
    return mm.Pg[g,t] - mm.Pg[g, mm.PERIOD.prev(t)] <= mm.gen_RRlimit[g]
def rr_dn(mm,g,t):
    if t == mm.PERIOD.first(): return Constraint.Skip
    return mm.Pg[g, mm.PERIOD.prev(t)] - mm.Pg[g,t] <= mm.gen_RRlimit[g]
m.genRR_Up = Constraint(m.GEN, m.PERIOD, rule=rr_up)
m.genRR_Dn = Constraint(m.GEN, m.PERIOD, rule=rr_dn)

def vu_rule(mm,g,t):
    if t == mm.PERIOD.first(): return mm.v[g,t] >= mm.u[g,t]
    return mm.v[g,t] >= mm.u[g,t] - mm.u[g, mm.PERIOD.prev(t)]
m.genVU = Constraint(m.GEN, m.PERIOD, rule=vu_rule)

m.ESLimit_Mode  = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.u_c_ES[e,t]+mm.u_d_ES[e,t] <= 1)
m.ESLimit_PMaxC = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.P_c_ES[e,t] <= mm.ES_PmaxC[e]*mm.u_c_ES[e,t])
m.ESLimit_PMaxD = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.P_d_ES[e,t] <= mm.ES_PmaxD[e]*mm.u_d_ES[e,t])
m.ESLimit_EMax  = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.E_ES[e,t] <= mm.ES_Emax[e])
m.ESLimit_EMin  = Constraint(m.STORAGE, m.PERIOD, rule=lambda mm,e,t: mm.ES_Emin[e] <= mm.E_ES[e,t])

def E_calc(mm,e,t):
    if t == mm.PERIOD.first():
        return mm.E_ES[e,t] == mm.ES_EInit[e] + mm.ES_EffiC[e]*mm.P_c_ES[e,t] - mm.P_d_ES[e,t]/mm.ES_EffiD[e]
    return mm.E_ES[e,t] == mm.E_ES[e, mm.PERIOD.prev(t)] + mm.ES_EffiC[e]*mm.P_c_ES[e,t] - mm.P_d_ES[e,t]/mm.ES_EffiD[e]
m.ES_Ecalc = Constraint(m.STORAGE, m.PERIOD, rule=E_calc)
m.ES_ESame = Constraint(m.STORAGE, rule=lambda mm,e: mm.E_ES[e, mm.PERIOD.last()] == mm.ES_EInit[e])

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print("g  t   v       u    Pg")
for g in m.GEN:
    for t in m.PERIOD:
        print(f"{g}  {t}   {value(m.v[g,t]):.2f}   {int(round(value(m.u[g,t])))}   {value(m.Pg[g,t]):.3f}")
print("\nStorage:")
for e in m.STORAGE:
    for t in m.PERIOD:
        print(f"  e={e} t={t}  P_c={value(m.P_c_ES[e,t]):.3f}  P_d={value(m.P_d_ES[e,t]):.3f}  E={value(m.E_ES[e,t]):.3f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmp6pr4v6ea.pyomo.lp


Reading time = 0.00 seconds
x1: 48 rows, 33 columns, 104 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90
MIPGap  0

Optimize a model with 48 rows, 33 columns and 104 nonzeros
Model fingerprint: 0x0a475c29
Variable types: 21 continuous, 12 integer (12 binary)
Coefficient statistics:


  Matrix range     [1e+00, 1e+02]
  Objective range  [6e-01, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+03]
Found heuristic solution: objective 321.0000000


Presolve removed 37 rows and 19 columns


Presolve time: 0.00s
Presolved: 11 rows, 14 columns, 32 nonzeros


...

H    0     0                     296.0000000  296.00000  0.00%     -    0s


     0     0  296.00000    0    2  296.00000  296.00000  0.00%     -    0s



Explored 1 nodes (11 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 20 (of 20 available processors)

Solution count 4: 296 301 309 321 



Optimal solution found (tolerance 0.00e+00)
Best objective 2.960000000000e+02, best bound 2.960000000000e+02, gap 0.0000%


ok optimal
g  t   v       u    Pg
1  1   1.00   1   120.000
1  2   0.00   1   120.000
1  3   0.00   1   120.000
2  1   0.00   0   0.000
2  2   1.00   1   40.000
2  3   0.00   0   0.000

Storage:
  e=1 t=1  P_c=0.000  P_d=10.000  E=40.000
  e=1 t=2  P_c=0.000  P_d=10.000  E=30.000
  e=1 t=3  P_c=20.000  P_d=0.000  E=50.000
